In [1]:
from spark_utils import get_spark_session, stop_spark_session

# Spark's built-in column functions
from pyspark.sql import functions as F

# Start a Spark session tagged for this notebook's job
spark = get_spark_session("SilverLinkingModel")

# Where cleaned Silver tables live (written by 02_silver_cleaning.ipynb)
SILVER_DIR = "../delta/silver"

D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# Read every Silver table.
# (Reading once up front, rather than re-reading inside each step below,
# keeps this notebook simple to follow top to bottom.)

orders = spark.read.format("delta").load(f"{SILVER_DIR}/olist_orders_dataset")
order_items = spark.read.format("delta").load(f"{SILVER_DIR}/olist_order_items_dataset")
order_payments = spark.read.format("delta").load(f"{SILVER_DIR}/olist_order_payments_dataset")
order_reviews = spark.read.format("delta").load(f"{SILVER_DIR}/olist_order_reviews_dataset")
customers = spark.read.format("delta").load(f"{SILVER_DIR}/olist_customers_dataset")
sellers = spark.read.format("delta").load(f"{SILVER_DIR}/olist_sellers_dataset")
products = spark.read.format("delta").load(f"{SILVER_DIR}/olist_products_dataset")
geolocation = spark.read.format("delta").load(f"{SILVER_DIR}/olist_geolocation_dataset")

print("All Silver tables loaded.")

All Silver tables loaded.


In [3]:
# --- Aggregate payments to one row per order_id ---

# An order can have multiple payment rows (e.g. split across installments or methods). I sum the value and count how many payment records existed,
# so joining this onto order_items later won't duplicate line items.

payments_agg = (
    order_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),           # total amount paid for the order
        F.count("payment_sequential").alias("payment_record_count"),   # how many payment records made up that total
    )
)

print(f"payments_agg: {payments_agg.count()} rows (one per distinct order_id)")

payments_agg: 99440 rows (one per distinct order_id)


In [4]:
# --- Aggregate reviews to one row per order_id ---

# An order occasionally has more than one review row in the raw data.
# I average the score (rare to differ, but avg handles it safely) and treat has_comment as True if ANY of that order's reviews had a comment.

reviews_agg = (
    order_reviews
    .groupBy("order_id")
    .agg(
        F.avg("review_score").alias("review_score"),          # average score if multiple reviews exist
        F.max("has_comment").alias("has_comment"),              # True if at least one review had written text
    )
)

print(f"reviews_agg: {reviews_agg.count()} rows (one per distinct order_id)")

reviews_agg: 99743 rows (one per distinct order_id)


In [5]:
# --- Build fact_order_items ---

# Grain: one row per product line item within an order.
# order_items joins to orders (1-to-1 on order_id side), then to the
# pre-aggregated payments and reviews (also 1-to-1 on order_id side),
# so no fan-out happens at any join step.

fact_order_items = (
    order_items
    .join(orders, on="order_id", how="left")            # bring in order_status, is_delivered, delivery_days, timestamps
    .join(payments_agg, on="order_id", how="left")      # bring in total_payment_value, payment_record_count
    .join(reviews_agg, on="order_id", how="left")       # bring in review_score, has_comment
)

print(f"fact_order_items: {fact_order_items.count()} rows")
print(f"order_items (source) had: {order_items.count()} rows")
# These two counts should match exactly - if fact_order_items is larger,
# a join fanned out somewhere and needs investigating before moving on.

fact_order_items: 112650 rows
order_items (source) had: 112650 rows


In [6]:
# --- Writing fact_order_items to Delta ---

fact_order_items.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/fact_order_items")

print(f"[Model] fact_order_items: {fact_order_items.count()} rows written")


[Model] fact_order_items: 112650 rows written


In [7]:
# --- dim_customers ---

# Left-join customers to geolocation (on zip prefix) to add lat/lng,
# so Power BI can map customers geographically without a separate join.

dim_customers = (
    customers
    .join(
        geolocation,
        customers["customer_zip_code_prefix"] == geolocation["geolocation_zip_code_prefix"],  # join on zip prefix
        how="left"                                                                            # keep all customers even without a geo match
    )
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "geolocation_lat",
        "geolocation_lng",
    )
)

dim_customers.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/dim_customers")

print(f"[Model] dim_customers: {dim_customers.count()} rows written")

[Model] dim_customers: 99441 rows written


In [8]:
# --- dim_sellers ---

# Same pattern as dim_customers: attaching lat/lng via the zip-prefix join.

dim_sellers = (
    sellers
    .join(
        geolocation,
        sellers["seller_zip_code_prefix"] == geolocation["geolocation_zip_code_prefix"],   # join on zip prefix
        how="left"                                                                         # keep all sellers even without a geo match
    )
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "geolocation_lat",
        "geolocation_lng",
    )
)

dim_sellers.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/dim_sellers")

print(f"[Model] dim_sellers: {dim_sellers.count()} rows written")

[Model] dim_sellers: 3095 rows written


In [9]:
# --- dim_products ---

# Already cleaned with an English category name in Silver - just select the columns the model actually needs and write it under its dim_ name.

dim_products = products.select(
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
)

dim_products.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/dim_products")

print(f"[Model] dim_products: {dim_products.count()} rows written")

[Model] dim_products: 32951 rows written


In [10]:
# --- dim_date ---

# Built fresh (not joined from another table): one row per calendar date spanning the full range of order purchase dates, with the breakdown
# columns Power BI needs for proper time intelligence.

# Find the min/max purchase date actually present in the data
date_range = orders.select(
    F.min(F.to_date("order_purchase_timestamp")).alias("min_date"),
    F.max(F.to_date("order_purchase_timestamp")).alias("max_date"),
).collect()[0]                                                          # collect() pulls this single small result back to the driver

min_date, max_date = date_range["min_date"], date_range["max_date"]
print(f"Order date range: {min_date} to {max_date}")

# Generate one row per day in that range using Spark's sequence() function, then explode the array into individual rows
dim_date = (
    spark.createDataFrame([(min_date, max_date)], ["start_date", "end_date"])
    .select(
        F.explode(
            F.sequence(F.col("start_date"), F.col("end_date"), F.expr("interval 1 day"))  # array of every date in the range
        ).alias("full_date")
    )
    .withColumn("year", F.year("full_date"))                       # e.g. 2017
    .withColumn("month", F.month("full_date"))                       # 1-12
    .withColumn("month_name", F.date_format("full_date", "MMMM"))      # e.g. "January"
    .withColumn("quarter", F.quarter("full_date"))                       # 1-4
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))         # e.g. "Monday"
    .withColumn("is_weekend", F.dayofweek("full_date").isin([1, 7]))         # True for Sat/Sun (Spark's dayofweek: 1=Sunday, 7=Saturday)
)

dim_date.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/dim_date")

print(f"[Model] dim_date: {dim_date.count()} rows written")

Order date range: 2016-09-04 to 2018-10-17
[Model] dim_date: 774 rows written


In [12]:
# Quick sanity check: previewing the fact table joined with a couple of its dimensions, to confirm the model actually links together correctly.

check_df = (
    fact_order_items
    .join(dim_products, on="product_id", how="left")
    .select(
        "order_id", "product_id", "product_category_name_english", "price", "total_payment_value", "review_score", "is_delivered",
    )
)

check_df.show(10, truncate=False)

+--------------------------------+--------------------------------+-----------------------------+------+-------------------+------------+------------+
|order_id                        |product_id                      |product_category_name_english|price |total_payment_value|review_score|is_delivered|
+--------------------------------+--------------------------------+-----------------------------+------+-------------------+------------+------------+
|00010242fe8c5a6d1ba2dd792cb16214|4244733e06e7ecb4970a6e2683c13e61|cool_stuff                   |58.9  |72.19              |5.0         |true        |
|00018f77f2f0320c557190d7a144bdd3|e5f2d52b802189ee658865ca93d83a8f|pet_shop                     |239.9 |259.83             |4.0         |true        |
|000229ec398224ef6ca0657da4fc703e|c777355d18b72b67abbeef9df44fd0fd|furniture_decor              |199.0 |216.87             |5.0         |true        |
|00024acbcdf0a6daa1e931b038114c75|7634da152a4610f1595efa32f14722fc|perfumery                  

In [13]:
stop_spark_session(spark)